In [ ]:
import os
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

DEVICE = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

BATCH_SIZE = 16
IMAGE_SIZE = 64
CHANNELS_IMG = 3

Z_DIM = 100

FEATURES_generator = 64
FEATURES_discriminator = 64

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BETA1 = 0.5

DATASET_NAME = "food101"
OUTPUT_DIR = "food_gan_outputs"


class Generator(keras.Model):

    def __init__(self, z_dim, channels_img, features_g):
        super().__init__()

        self.net = keras.Sequential([

            keras.layers.Conv2DTranspose(
                features_g * 16,
                kernel_size=4,
                strides=1,
                padding="valid",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.ReLU(),

            keras.layers.Conv2DTranspose(
                features_g * 8,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.ReLU(),

            keras.layers.Conv2DTranspose(
                features_g * 4,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.ReLU(),

            keras.layers.Conv2DTranspose(
                features_g * 2,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.ReLU(),

            keras.layers.Conv2DTranspose(
                channels_img,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.Activation("tanh")

        ])

    def call(self, x, training=False):
        return self.net(x, training=training)


class Discriminator(keras.Model):

    def __init__(self, channels_img, features_d):
        super().__init__()

        self.net = keras.Sequential([

            keras.layers.Conv2D(
                features_d,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=True,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.LeakyReLU(negative_slope=0.2),

            keras.layers.Conv2D(
                features_d * 2,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.LeakyReLU(negative_slope=0.2),

            keras.layers.Conv2D(
                features_d * 4,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.LeakyReLU(negative_slope=0.2),

            keras.layers.Conv2D(
                features_d * 8,
                kernel_size=4,
                strides=2,
                padding="same",
                use_bias=False,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            ),
            keras.layers.BatchNormalization(
                gamma_initializer=keras.initializers.RandomNormal(1.0, 0.02)
            ),
            keras.layers.LeakyReLU(negative_slope=0.2),

            keras.layers.Conv2D(
                1,
                kernel_size=4,
                strides=1,
                padding="valid",
                use_bias=True,
                kernel_initializer=keras.initializers.RandomNormal(0.0, 0.02)
            )

        ])

    def call(self, x, training=False):
        return self.net(x, training=training)


def main():

    print(f"Using Device: {DEVICE}")

    print("\nDownloading / Loading Food-101 Dataset...")

    train_dataset = tfds.load(
        DATASET_NAME,
        split="train",
        as_supervised=True
    )

    def preprocess_image(image, label):

        image = tf.image.resize(
            image,
            [IMAGE_SIZE, IMAGE_SIZE]
        )

        image = tf.cast(image, tf.float32)

        image = (image / 127.5) - 1.0

        return image

    dataloader = train_dataset.map(
        preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataloader = dataloader.shuffle(
        10000
    ).batch(
        BATCH_SIZE,
        drop_remainder=True
    ).prefetch(
        tf.data.AUTOTUNE
    )

    generator = Generator(
        Z_DIM,
        CHANNELS_IMG,
        FEATURES_generator
    )

    discriminator = Discriminator(
        CHANNELS_IMG,
        FEATURES_discriminator
    )

    generator.build((None, 1, 1, Z_DIM))
    discriminator.build((None, IMAGE_SIZE, IMAGE_SIZE, CHANNELS_IMG))

    opt_generator = keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        beta_1=BETA1,
        beta_2=0.999
    )

    opt_discriminator = keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        beta_1=BETA1,
        beta_2=0.999
    )

    criterion = keras.losses.BinaryCrossentropy(
        from_logits=True
    )

    fixed_noise = tf.random.normal(
        [32, 1, 1, Z_DIM]
    )

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    print("\nDataset compiled successfully!")
    print("Detected 101 Food Categories.")
    print(f"Starting training on {DEVICE}...\n")

    @tf.function
    def train_step(real_imgs):

        batch_size = tf.shape(real_imgs)[0]

        real_labels = tf.ones(
            [batch_size, 1, 1, 1]
        ) * 0.9

        fake_labels = tf.zeros(
            [batch_size, 1, 1, 1]
        )

        noise = tf.random.normal(
            [batch_size, 1, 1, Z_DIM]
        )

        with tf.GradientTape() as disc_tape:

            fake_imgs = generator(
                noise,
                training=True
            )

            output_real = discriminator(
                real_imgs,
                training=True
            )

            output_fake = discriminator(
                tf.stop_gradient(fake_imgs),
                training=True
            )

            loss_discriminator_real = criterion(
                real_labels,
                output_real
            )

            loss_discriminator_fake = criterion(
                fake_labels,
                output_fake
            )

            loss_discriminator = (
                loss_discriminator_real +
                loss_discriminator_fake
            )

        disc_gradients = disc_tape.gradient(
            loss_discriminator,
            discriminator.trainable_variables
        )

        opt_discriminator.apply_gradients(
            zip(
                disc_gradients,
                discriminator.trainable_variables
            )
        )

        with tf.GradientTape() as gen_tape:

            fake_imgs = generator(
                noise,
                training=True
            )

            output_fake_for_generator = discriminator(
                fake_imgs,
                training=True
            )

            loss_generator = criterion(
                tf.ones_like(output_fake_for_generator),
                output_fake_for_generator
            )

        gen_gradients = gen_tape.gradient(
            loss_generator,
            generator.trainable_variables
        )

        opt_generator.apply_gradients(
            zip(
                gen_gradients,
                generator.trainable_variables
            )
        )

        return loss_discriminator, loss_generator

    for epoch in range(NUM_EPOCHS):

        for batch_idx, real_imgs in enumerate(dataloader):

            with tf.device(DEVICE):

                loss_discriminator, loss_generator = train_step(
                    real_imgs
                )

            if batch_idx % 10 == 0:

                print(
                    f"Epoch {epoch + 1} | Batch {batch_idx} "
                    f"Loss Discriminator : {loss_discriminator.numpy():.4f}, "
                    f"Loss Generator : {loss_generator.numpy():.4f}"
                )

        fake_snapshot = generator(
            fixed_noise,
            training=False
        )

        fake_snapshot = (fake_snapshot + 1.0) / 2.0

        fake_snapshot = tf.clip_by_value(
            fake_snapshot,
            0.0,
            1.0
        )

        grid = tf.reshape(
            tf.transpose(
                tf.reshape(
                    fake_snapshot,
                    [4, 8, IMAGE_SIZE, IMAGE_SIZE, CHANNELS_IMG]
                ),
                [0, 2, 1, 3, 4]
            ),
            [4 * IMAGE_SIZE, 8 * IMAGE_SIZE, CHANNELS_IMG]
        )

if __name__ == "__main__":
    main()